# Inferring the hydrostatic mass bias $B$ from catalogue $Y_{5R_{500c}}$

We infer the effective hydrostatic mass bias $B \equiv M_{500c}^{\rm true}/M_{500c}^{\rm HSE} = (1-b)^{-1}$
**purely from the officially-provided** spherical Compton-$Y$, $Y_{5R_{500c}}$, by assuming the
Arnaud et al. (2010, A10) GNFW pressure profile.

## Key observation about the catalogue column

The SOAP aperture quantity stored as `Y_5R500c_Mpc2` is the **intrinsic / spherical** Compton-$Y$,
i.e. $D_A^2(z)\, Y_{\rm sph}(<5R_{500c})$, an area in $\mathrm{Mpc}^2$. It already includes the
$D_A^2$ factor, so the inference needs **no separate angular-diameter distance**: the catalogue
column *is* $D_A^2 Y_{\rm obs}$.

## A10 prediction

Substituting the A10 GNFW profile into the self-similar scaling (see derivation),
$$
D_A^2(z)\, Y_{5R_{500c}}(M_{\rm true}, z; B) = \frac{\sigma_T}{m_e c^2}\, 4\pi\, P_0\, I_5^{\rm GNFW}\,
R_{500c}^3\!\left(\tfrac{M_{\rm true}}{B}, z\right)\, P_{500}\!\left(\tfrac{M_{\rm true}}{B}, z\right),
$$
with the dimensionless GNFW integral $I_5^{\rm GNFW}=\int_0^5 x^2 p(x)\,dx$ and the A10 (empirical-tilt)
normalization $P_{500}\propto (M/B)^{2/3+\alpha_p} E(z)^{8/3}$, $\alpha_p=0.12$. Since
$R_{500c}^3 P_{500}\propto (M/B)^{5/3+\alpha_p} E(z)^{2/3}$, defining $\gamma_Y\equiv 5/3+\alpha_p\simeq1.79$,
$$
Y_{5R_{500c}}^{\rm Mpc^2}(M,z;B) = Y_{5R_{500c}}^{\rm Mpc^2}(M,z;B{=}1)\; B^{-\gamma_Y}.
$$

## Per-cluster inference

With the catalogue providing $Y_{\rm obs}\equiv Y_{5R_{500c}}^{\rm Mpc^2}$ and the true mass $M_{500c}$,
$$
\boxed{\;B = \left[\frac{Y_{5R_{500c}}^{\rm Mpc^2}(M,z;B{=}1)}{Y_{\rm obs}}\right]^{1/\gamma_Y}\;}
$$
We use hmfast's exact A10 constants (the `GNFWPressureProfile` defaults, which hard-code the
$2/3+0.12$ tilt) so the normalization is consistent with the rest of the pipeline.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Publication-quality plot defaults
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

from scipy.integrate import quad

from flamingo import paths
from flamingo.catalogue import load_catalogue, frame
from hmfast.halos.profiles.pressure import GNFWPressureProfile

# Officially-provided (hdfstream / SOAP) spherical Y_5R500c lives in the raw
# catalogue; the y0q file carries the same column plus z, M_500c, R_500c.
CAT = paths.HYDRO / 'catalogue' / 'halo_catalogue_M500c_5e13_zlt3_y0q_arnaudB1.csv'
df = load_catalogue(CAT)
df = df[(df['Y_5R500c_Mpc2'] > 0) & (df['M_500c_Msun'] > 0) & (df['R_500c_Mpc'] > 0)]
print('clusters:', len(df))
print(df[['z', 'M_500c_Msun', 'R_500c_Mpc', 'Y_5R500c_Mpc2']].describe().round(6).to_string())

z = df['z'].values
M = df['M_500c_Msun'].values            # true M_500c (simulation SO mass)
R = df['R_500c_Mpc'].values             # proper Mpc
Yobs = df['Y_5R500c_Mpc2'].values       # = D_A^2 Y_sph(<5R500c), proper Mpc^2
Ez = frame.efunc(z)                     # E(z)=H(z)/H0, FLAMINGO D3A cosmology
h = frame._h                            # 0.681


clusters: 1555542
                  z   M_500c_Msun    R_500c_Mpc  Y_5R500c_Mpc2
count  1.555542e+06  1.555542e+06  1.555542e+06   1.555542e+06
mean   8.891080e-01  8.902150e+13  4.908960e-01   7.000000e-06
std    4.404630e-01  5.834199e+13  1.226800e-01   1.000000e-05
min    3.211000e-03  5.008000e+13  2.104690e-01   1.000000e-06
25%    5.609740e-01  5.784000e+13  4.082590e-01   3.000000e-06
50%    8.274470e-01  7.048000e+13  4.753730e-01   4.000000e-06
75%    1.153245e+00  9.696000e+13  5.506740e-01   7.000000e-06
max    2.999426e+00  2.065920e+15  1.823297e+00   6.470000e-04


I0000 00:00:1782460114.070443 2538684 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782460114.070792 2538684 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1782460114.989690 2538684 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782460114.989933 2538684 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


## GNFW shape integral and the A10 normalization

$I_5^{\rm GNFW}$ uses the hmfast A10 default shape ($P_0=8.130$, $c_{500}=1.156$, $\alpha=1.062$,
$\beta=5.481$, $\gamma=0.329$), with $P_0$ factored out (it multiplies the amplitude separately).
$P_{500}$ uses the same constants and units as `GNFWPressureProfile` (the A10 prefactor evaluates to
$\mathrm{eV\,cm^{-3}}$). We compute the proper $R_{500c}$ from the spherical-overdensity definition,
$M=\tfrac{4\pi}{3}\,500\,\rho_c(z)\,R_{500c}^3$ with $\rho_c(z)=\rho_{c,0}E^2(z)$.

**Sanity check:** $R_{500c}$ recomputed from $M_{500c}$ must reproduce the catalogue $R_{500c}$
(confirms the proper-radius convention and $\rho_{c,0}$).


In [2]:
prof = GNFWPressureProfile(B=1.0)           # A10 defaults
P0, c500, al, be, ga = prof.P0, prof.c500, prof.alpha, prof.beta, prof.gamma

def p_shape(x):
    xc = c500 * x
    return xc**(-ga) * (1.0 + xc**al)**((ga - be) / al)

I5 = quad(lambda x: p_shape(x) * x * x, 0.0, 5.0, limit=400)[0]
print(f'GNFW shape params: P0={P0}, c500={c500}, alpha={al}, beta={be}, gamma={ga}')
print(f'I5^GNFW = int_0^5 x^2 p(x) dx = {I5:.6e}')

# Physical constants (match hmfast pressure.py)
SIGMA_T = 6.6524587e-25       # cm^2
MEC2_EV = 510998.95           # eV
MPC_CM  = 3.0856775814913673e24
RHO_C0  = 2.775e11 * h * h     # Msun / Mpc^3 (proper, z=0)
ALPHA_P = 0.12
GAMMA_Y = 5.0 / 3.0 + ALPHA_P  # 1.7867

def R500_of_M(Mmass, zz):
    # Proper R_500c [Mpc] from spherical-overdensity mass.
    return (3.0 * Mmass / (4.0 * np.pi * 500.0 * RHO_C0 * frame.efunc(zz)**2))**(1.0 / 3.0)

# --- sanity: R500 from M vs catalogue R500 ---
R_from_M = R500_of_M(M, z)
ratio = R / R_from_M
print(f'R500 catalogue / R500(M): median={np.median(ratio):.5f}  std={np.std(ratio):.5f}')
assert abs(np.median(ratio) - 1.0) < 0.01, 'rho_c convention mismatch'

def Y_A10_Mpc2(Mmass, zz, B):
    # A10 spherical Y_5R500c (= D_A^2 Y_sph) in proper Mpc^2 at bias B.
    Ezz = frame.efunc(zz)
    Mt = Mmass / B                                   # HSE mass entering A10
    R500_cm = R500_of_M(Mt, zz) * MPC_CM             # proper cm
    P500 = (1.65 * (h / 0.7)**2 * Ezz**(8.0 / 3.0)
            * (Mt * h / (0.7 * 3e14))**(2.0 / 3.0 + ALPHA_P)
            * (0.7 / h)**1.5)                        # eV / cm^3
    Y_cm2 = (SIGMA_T / MEC2_EV) * 4.0 * np.pi * P0 * P500 * R500_cm**3 * I5
    return Y_cm2 / MPC_CM**2


GNFW shape params: P0=8.13, c500=1.156, alpha=1.062, beta=5.4807, gamma=0.3292
I5^GNFW = int_0^5 x^2 p(x) dx = 4.874882e-02
R500 catalogue / R500(M): median=0.99983  std=0.00305


## Infer $B$ per cluster

Compute the $B{=}1$ A10 prediction $Y_1(M,z)$ and invert
$Y_{\rm obs}=Y_1\,B^{-\gamma_Y}\Rightarrow B=(Y_1/Y_{\rm obs})^{1/\gamma_Y}$.


In [3]:
Y1 = Y_A10_Mpc2(M, z, 1.0)
B_infer = (Y1 / Yobs)**(1.0 / GAMMA_Y)
df = df.assign(B_infer=B_infer)

print(f'median Y_obs / Y_A10(B=1) = {np.median(Yobs / Y1):.4f}')
print(f'inferred B: median={np.median(B_infer):.4f}  mean={np.mean(B_infer):.4f}')
print(f'           16/50/84 pct = {np.percentile(B_infer, [16, 50, 84]).round(4)}')
for mcut, lbl in [(1e14, 'M>1e14'), (2e14, 'M>2e14'), (5e14, 'M>5e14')]:
    s = M > mcut
    print(f'  {lbl:8s} (N={int(s.sum()):>7d}): median B = {np.median(B_infer[s]):.4f}'
          f'  16/84 = {np.percentile(B_infer[s], [16, 84]).round(3)}')


median Y_obs / Y_A10(B=1) = 0.9834
inferred B: median=1.0094  mean=1.0044
           16/50/84 pct = [0.8626 1.0094 1.1507]


  M>1e14   (N= 361536): median B = 1.0690  16/84 = [0.932 1.204]
  M>2e14   (N=  64647): median B = 1.1387  16/84 = [1.013 1.271]
  M>5e14   (N=   3738): median B = 1.2488  16/84 = [1.124 1.39 ]


## Figure 1: inferred $B$ vs mass

$B$ is degenerate with the A10 amplitude, so the absolute level reflects the A10 calibration;
the informative quantity is the **mass trend**. At low mass A10 ($B{=}1$) slightly over-predicts
the FLAMINGO $Y$ (so $B\lesssim1$), while massive clusters require $B\sim1.1$-$1.2$.


In [4]:
fig, ax = plt.subplots(figsize=(7.2, 5.4))
hb = ax.hexbin(M, B_infer, xscale='log', gridsize=70, bins='log', cmap='viridis', mincnt=1)
cb = fig.colorbar(hb, ax=ax); cb.set_label(r'$\log_{10} N_{\rm clusters}$')

edges = np.logspace(np.log10(M.min()), np.log10(M.max()), 22)
cen = np.sqrt(edges[:-1] * edges[1:])
med = np.array([np.median(B_infer[(M >= edges[i]) & (M < edges[i+1])])
                if np.any((M >= edges[i]) & (M < edges[i+1])) else np.nan
                for i in range(len(cen))])
lo = np.array([np.percentile(B_infer[(M >= edges[i]) & (M < edges[i+1])], 16)
               if np.sum((M >= edges[i]) & (M < edges[i+1])) > 5 else np.nan for i in range(len(cen))])
hi = np.array([np.percentile(B_infer[(M >= edges[i]) & (M < edges[i+1])], 84)
               if np.sum((M >= edges[i]) & (M < edges[i+1])) > 5 else np.nan for i in range(len(cen))])
ax.plot(cen, med, color='#d62728', lw=2.4, label='running median')
ax.fill_between(cen, lo, hi, color='#d62728', alpha=0.25, label='16-84 pct')
ax.axhline(1.0, color='k', ls='--', lw=1.2, label=r'$B=1$ (no bias)')
ax.axhline(1.4, color='0.4', ls=':', lw=1.2, label=r'$B=1.4$ (Planck)')

ax.set_xlabel(r'$M_{500c}\ [M_\odot]$', fontsize=12)
ax.set_ylabel(r'inferred $B = M_{\rm true}/M_{\rm HSE}$', fontsize=12)
ax.set_ylim(0.6, 1.8)
ax.set_title('Hydrostatic bias inferred from catalogue $Y_{5R500c}$ (A10 GNFW)', fontsize=12)
ax.legend(fontsize=10, loc='upper left')
ax.tick_params(labelsize=10)
fig.tight_layout()
fig.savefig('../autoresearch/figures/nb13_B_vs_mass.pdf')
fig.savefig('../autoresearch/figures/nb13_B_vs_mass.png', dpi=300)
plt.show()


## Figure 2: $Y$-$M$ relation with A10 overlays

The de-evolved scaling $D_A^2 Y\,E^{-2/3}$ vs $M_{500c}$. Overplotted are the A10 prediction at
$B{=}1$ and at the sample-median inferred $B$. Because the A10 mass exponent is $\gamma_Y\simeq1.79$
while a $B$ shift is a pure vertical (mass-independent) translation in this plane, a single constant
$B$ cannot perfectly track the data slope, which is what drives the mass dependence of the inferred $B$.


In [5]:
Yde = Yobs / Ez**(2.0 / 3.0)
fig, ax = plt.subplots(figsize=(7.2, 5.4))
hb = ax.hexbin(M, Yde, xscale='log', yscale='log', gridsize=70, bins='log', cmap='cividis', mincnt=1)
cb = fig.colorbar(hb, ax=ax); cb.set_label(r'$\log_{10} N_{\rm clusters}$')

Mgrid = np.logspace(np.log10(M.min()), np.log10(M.max()), 60)
zmed = np.median(z)
y1_grid = Y_A10_Mpc2(Mgrid, np.full_like(Mgrid, zmed), 1.0) / frame.efunc(zmed)**(2.0 / 3.0)
Bmed = np.median(B_infer)
yB_grid = Y_A10_Mpc2(Mgrid, np.full_like(Mgrid, zmed), Bmed) / frame.efunc(zmed)**(2.0 / 3.0)
ax.plot(Mgrid, y1_grid, 'w-', lw=3.2)
ax.plot(Mgrid, y1_grid, color='#d62728', lw=2.0, label=r'A10 $B=1$')
ax.plot(Mgrid, yB_grid, color='#1f77b4', lw=2.0, ls='--', label=fr'A10 $B={Bmed:.2f}$ (median)')

ax.set_xlabel(r'$M_{500c}\ [M_\odot]$', fontsize=12)
ax.set_ylabel(r'$D_A^2 Y_{5R500c}\,E(z)^{-2/3}\ [\mathrm{Mpc}^2]$', fontsize=12)
ax.set_title(fr'$Y$-$M$ relation vs A10 (evaluated at $z={zmed:.2f}$)', fontsize=12)
ax.legend(fontsize=10, loc='upper left')
ax.tick_params(labelsize=10)
fig.tight_layout()
fig.savefig('../autoresearch/figures/nb13_YM_A10.pdf')
fig.savefig('../autoresearch/figures/nb13_YM_A10.png', dpi=300)
plt.show()


## Caveats

- $B$ is **degenerate with the A10 amplitude** ($P_0 I_5$) and with cosmology; the *absolute* level is
  only as trustworthy as the A10 calibration. The robust output is the **mass dependence** of the
  inferred bias.
- The inversion is exact and deterministic per cluster (no fit); intrinsic $Y$ scatter at fixed mass
  maps directly into scatter in $B$.
- We assumed the A10 aperture ($5R_{500c}$) matches the catalogue aperture; both integrate to
  $5R_{500c}$, so this is consistent.
